# 🌡️ KNYC Temperature Nowcaster — v2

Three models, all XGBoost:
- **Model A** — Daily high (settles to NWS CLI)
- **Model B** — t+3h temperature
- **Model C** — t+6h temperature

Trained on raw KNYC + upstream-station ASOS data with NY-local-time features
and proper early stopping. Run `build_training_data.py` first to produce
`knyc_training.csv`.

## 0. Imports & Config

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

DATA_PATH = 'knyc_training.csv'
RANDOM_STATE = 42
# Only stations sharing KNYC's :51 cadence land in the current CSV.

UPSTREAM = ['KEWR', 'KLGA', 'KJFK', 'KTEB', 'KPHL', 'KBWI', 'KDCA', 'KBOS']
print('Imports OK')

## 1. Load Data

In [6]:
df = pd.read_csv(DATA_PATH, parse_dates=['valid'])
df['valid'] = pd.to_datetime(df['valid'], utc=True)
df = df.sort_values('valid').reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df.valid.min()} → {df.valid.max()}')
print(f'CLI target coverage: {df["target_high"].notna().mean():.1%}')
df[['tmpf','dwpf','mslp','sknt','relh']].describe()

Shape: (140233, 114)
Date range: 2010-01-01 00:51:00+00:00 → 2026-01-31 23:51:00+00:00
CLI target coverage: 100.0%


,tmpf,dwpf,mslp,sknt,relh
count,139979.000000,139945.000000,138002.000000,127183.000000,139887.000000
mean,56.146574,41.631381,1016.471258,4.554570,60.966531
std,17.517630,19.362836,7.852567,3.247815,19.387933
min,-1.000000,-19.000000,966.000000,0.000000,7.200000
25%,42.000000,27.000000,1011.600000,3.000000,46.090000
50%,57.000000,43.000000,1016.400000,4.000000,59.400000
75%,71.000000,58.000000,1021.500000,6.000000,76.160000
max,103.000000,79.000000,1048.300000,74.000000,100.000000


## 2. Feature Columns

All raw, unscaled. Tree models do not need feature scaling.

In [7]:
FEATURE_COLS = [
    # Raw obs
    'tmpf', 'dwpf', 'relh', 'mslp', 'sknt', 'feel_gap',
    'u_wind', 'v_wind',
    # Derived met
    'dew_depression', 'dew_dep_tend_3h',
    'cloud_oktas', 'cloud_tend_3h', 'ceiling_ft',
    'p01i', 'precip_24h', 'is_precip',
    # Solar geometry
    'solar_zenith', 'solar_alt', 'cos_zenith',
    # Climatology anchor
    'tmpf_clim_global', 'tmpf_anomaly',
    # Tendencies
    'mslp_tend_1h', 'mslp_tend_3h', 'mslp_tend_6h',
    'tmpf_tend_1h', 'tmpf_tend_3h', 'tmpf_tend_6h',
    # Lags
    *[f'tmpf_lag_{l}h' for l in [1, 2, 3, 6, 12, 24]],
    *[f'dwpf_lag_{l}h' for l in [1, 2, 3, 6, 12, 24]],
    # Rolling
    'tmpf_roll3_mean', 'tmpf_roll6_mean', 'tmpf_roll24_mean',
    'tmpf_roll3_std', 'mslp_roll3_mean',
    # Time
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'hour', 'month', 'dayofyear',
]

# Upstream station features
for code in UPSTREAM:
    for suffix in ('tmpf', 'tmpf_delta', 'tmpf_tend_3h', 'sknt', 'drct'):
        col = f'{code}_{suffix}'
        if col in df.columns:
            FEATURE_COLS.append(col)

print(f'{len(FEATURE_COLS)} feature columns')
missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    print(f'MISSING from data: {missing}')
    FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

91 feature columns


## 3. Train / Val / Test Split

Time-aware. No shuffling.
- Train: 2010–2021
- Val:   2022–2023
- Test:  2024+

In [ ]:
df_clean = df.dropna(subset=FEATURE_COLS + ['target_high', 'target_t3h', 'target_t6h']).copy()
print(f'Rows after dropna: {len(df_clean):,}')

if len(df_clean) == 0:
    na_pct = df[FEATURE_COLS + ['target_high','target_t3h','target_t6h']].isna().mean()
    worst = na_pct.sort_values(ascending=False).head(15)
    raise RuntimeError(
        'No rows survived dropna. Worst-coverage columns:\n' + worst.to_string()
    )

year = df_clean['valid'].dt.year
train = df_clean[year <= 2021].copy()
val   = df_clean[(year >= 2022) & (year <= 2023)].copy()
test  = df_clean[year >= 2024].copy()

assert len(train) > 0, 'Train split empty'
assert len(val)   > 0, 'Val split empty'
assert len(test)  > 0, 'Test split empty'
assert train['valid'].dt.year.max() <= 2021, 'Train contains data past 2021'
assert val['valid'].dt.year.min() >= 2022,   'Val starts before 2022'
assert test['valid'].dt.year.min() >= 2024,  'Test starts before 2024'

print(f'Train: {len(train):,} ({train.valid.min().date()} → {train.valid.max().date()})')
print(f'Val:   {len(val):,} ({val.valid.min().date()} → {val.valid.max().date()})')
print(f'Test:  {len(test):,} ({test.valid.min().date()} → {test.valid.max().date()})')

# Recompute climatology from TRAIN slice only.
clim_lookup = (train.groupby(['month','hour'])['tmpf']
                    .mean()
                    .rename('tmpf_clim_train'))
for split in (train, val, test):
    key_idx = pd.MultiIndex.from_arrays(
        [split['month'].astype(int), split['hour'].astype(int)])
    split['tmpf_clim_global'] = key_idx.map(clim_lookup).astype(float)
    split['tmpf_clim_global'] = split['tmpf_clim_global'].fillna(split['tmpf'])
    split['tmpf_anomaly'] = split['tmpf'] - split['tmpf_clim_global']

clim_lookup.reset_index().rename(columns={'tmpf_clim_train':'tmpf_clim'}).to_csv(
    'tmpf_climatology.csv', index=False)

min_year = train['valid'].dt.year.min()
sample_weight = np.exp(0.10 * (train['valid'].dt.year - min_year))

X_train = train[FEATURE_COLS]
X_val   = val[FEATURE_COLS]
X_test  = test[FEATURE_COLS]

## 4. Persistence Baselines (so we know what we're beating)

In [ ]:
# t+3h / t+6h persistence = forecast = current temp.
pers_t3h_mae = mean_absolute_error(test['target_t3h'], test['tmpf'])
pers_t6h_mae = mean_absolute_error(test['target_t6h'], test['tmpf'])

# Daily-high persistence = today's high = yesterday's CLI high.
test_sorted = test.sort_values('valid')
daily = (test_sorted.groupby(test_sorted['valid'].dt.date)
                    .agg(target=('target_high','first'))
                    .reset_index())
daily['yesterday'] = daily['target'].shift(1)
pers_high_mae = mean_absolute_error(
    daily['target'].dropna(),
    daily['yesterday'].fillna(daily['target']).loc[daily['target'].notna()]
)

# Climatology baseline for daily high.
clim_high = train.groupby(train['valid'].dt.dayofyear)['target_high'].mean()
clim_pred = test['valid'].dt.dayofyear.map(clim_high)
clim_high_mae = mean_absolute_error(test['target_high'], clim_pred)

print('── Baselines (test set) ──')
print(f'  Persistence t+3h MAE: {pers_t3h_mae:.2f}°F')
print(f'  Persistence t+6h MAE: {pers_t6h_mae:.2f}°F')
print(f'  Persistence high MAE: {pers_high_mae:.2f}°F')
print(f'  Climatology high MAE: {clim_high_mae:.2f}°F')

## 5. Model A — Daily High

In [ ]:
y_train_high = train['target_high']
y_val_high   = val['target_high']
y_test_high  = test['target_high']

model_high = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=20,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    early_stopping_rounds=50,
    objective='reg:squarederror',
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
model_high.fit(
    X_train, y_train_high,
    sample_weight=sample_weight,
    eval_set=[(X_val, y_val_high)],
    verbose=False,
)
print(f'Best iteration: {model_high.best_iteration}')

pred_val_high  = model_high.predict(X_val)
pred_test_high = model_high.predict(X_test)

mae_val_high  = mean_absolute_error(y_val_high, pred_val_high)
rmse_val_high = mean_squared_error(y_val_high, pred_val_high) ** 0.5
mae_test_high = mean_absolute_error(y_test_high, pred_test_high)
rmse_test_high = mean_squared_error(y_test_high, pred_test_high) ** 0.5

print('── Model A: Daily High ──')
print(f'  Val  MAE: {mae_val_high:.2f}°F  RMSE: {rmse_val_high:.2f}°F')
print(f'  Test MAE: {mae_test_high:.2f}°F  RMSE: {rmse_test_high:.2f}°F')
print(f'  vs persistence: {pers_high_mae - mae_test_high:+.2f}°F improvement')
print(f'  vs climatology: {clim_high_mae - mae_test_high:+.2f}°F improvement')

# Bias check (warm bias was a flagged weakness).
bias = (pred_test_high - y_test_high).mean()
print(f'  Bias: {bias:+.2f}°F (positive = warm bias)')

## 6. Model B / C — t+3h and t+6h

In [ ]:
results = {}
for horizon, target_col, pers_mae in [('t+3h', 'target_t3h', pers_t3h_mae),
                                       ('t+6h', 'target_t6h', pers_t6h_mae)]:
    y_tr = train[target_col]; y_va = val[target_col]; y_te = test[target_col]
    model = XGBRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=20,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        early_stopping_rounds=50,
        objective='reg:squarederror',
        tree_method='hist',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_train, y_tr,
              sample_weight=sample_weight,
              eval_set=[(X_val, y_va)],
              verbose=False)
    pred_val = model.predict(X_val); pred_test = model.predict(X_test)
    mae_v = mean_absolute_error(y_va, pred_val)
    rmse_v = mean_squared_error(y_va, pred_val) ** 0.5
    mae_t = mean_absolute_error(y_te, pred_test)
    rmse_t = mean_squared_error(y_te, pred_test) ** 0.5
    print(f'── Model {horizon} ──')
    print(f'  Best iter: {model.best_iteration}')
    print(f'  Val  MAE: {mae_v:.2f}°F  RMSE: {rmse_v:.2f}°F')
    print(f'  Test MAE: {mae_t:.2f}°F  RMSE: {rmse_t:.2f}°F')
    print(f'  vs persistence ({pers_mae:.2f}°F): {pers_mae - mae_t:+.2f}°F improvement')
    results[horizon] = dict(model=model, pred_val=pred_val, pred_test=pred_test,
                            y_val=y_va, y_test=y_te,
                            mae_val=mae_v, rmse_val=rmse_v,
                            mae_test=mae_t, rmse_test=rmse_t)

## 7. Diagnostic Plots

In [1]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
configs = [
    ('Daily High', pred_test_high, y_test_high.values, '#e63946'),
    ('t+3h', results['t+3h']['pred_test'], results['t+3h']['y_test'].values, '#457b9d'),
    ('t+6h', results['t+6h']['pred_test'], results['t+6h']['y_test'].values, '#2a9d8f'),
]
for ax, (title, pred, actual, color) in zip(axes, configs):
    ax.scatter(actual, pred, alpha=0.15, s=3, color=color)
    lims = [min(actual.min(), pred.min()) - 2, max(actual.max(), pred.max()) + 2]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_title(f'{title}  MAE={mean_absolute_error(actual,pred):.2f}°F', fontweight='bold')
    ax.set_xlabel('Actual (°F)'); ax.set_ylabel('Predicted (°F)')
plt.suptitle('Predicted vs Actual — Test Set', y=1.02)
plt.tight_layout(); plt.show()

NameError: name 'plt' is not defined

In [ ]:
# Error by NY-local hour and month.
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for col, (title, pred, actual, color) in enumerate(configs):
    abs_err = np.abs(np.asarray(actual) - np.asarray(pred))
    err = pd.DataFrame({
        'hour': test['hour'].values,
        'month': test['month'].values,
        'abs_err': abs_err,
    })
    by_hour = err.groupby('hour')['abs_err'].mean()
    by_month = err.groupby('month')['abs_err'].mean()
    axes[0, col].bar(by_hour.index, by_hour.values, color=color, alpha=0.85)
    axes[0, col].set_title(f'{title} — MAE by Local Hour', fontweight='bold')
    axes[0, col].set_xlabel('Hour (NY local)'); axes[0, col].set_ylabel('MAE (°F)')
    axes[1, col].bar(by_month.index, by_month.values, color=color, alpha=0.85)
    axes[1, col].set_title(f'{title} — MAE by Month', fontweight='bold')
    axes[1, col].set_xlabel('Month'); axes[1, col].set_ylabel('MAE (°F)')
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance (permutation, on val — gain can be misleading).
print('Computing permutation importance for t+3h (slow)...')
rng = np.random.default_rng(RANDOM_STATE)
subset_idx = rng.choice(len(X_val), size=min(3000, len(X_val)), replace=False)
perm = permutation_importance(
    results['t+3h']['model'], X_val.iloc[subset_idx], val['target_t3h'].iloc[subset_idx],
    n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1, scoring='neg_mean_absolute_error',
)
imp = pd.Series(perm.importances_mean, index=FEATURE_COLS).sort_values(ascending=True).tail(25)
fig, ax = plt.subplots(figsize=(8, 8))
imp.plot.barh(ax=ax, color='#457b9d', alpha=0.85)
ax.set_title('Top 25 Features — Permutation Importance (t+3h)', fontweight='bold')
ax.set_xlabel('Δ MAE when feature shuffled')
plt.tight_layout(); plt.show()

## 8. Persist Models + Feature Manifest

In [ ]:
joblib.dump(model_high, 'knyc_model_daily_high.pkl')
joblib.dump(results['t+3h']['model'], 'knyc_model_t3h.pkl')
joblib.dump(results['t+6h']['model'], 'knyc_model_t6h.pkl')

# Save feature manifest — the bot loads this to guarantee column order matches.
manifest = {
    'feature_cols': FEATURE_COLS,
    'upstream_stations': UPSTREAM,
    'train_through': str(train['valid'].max()),
    'test_metrics': {
        'daily_high_mae': float(mae_test_high),
        't3h_mae': float(results['t+3h']['mae_test']),
        't6h_mae': float(results['t+6h']['mae_test']),
    },
}
with open('feature_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print('✅ Models + manifest saved')